<a href="https://colab.research.google.com/github/Nishal-01/Nishaltejreddy_INFO5731_Spring2026/blob/main/Chennu_Nishal_Tej_Reddy_Assignment_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 Assignment 1**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


In [1]:
# QUESTION 1

!pip install requests pandas

import requests
import pandas as pd
import time

API_URL = "https://api.semanticscholar.org/graph/v1/paper/search"

headers = {
    "User-Agent": "Mozilla/5.0"
}

queries = [
    "machine learning",
    "artificial intelligence",
    "data science",
    "information extraction"
]

limit_per_request = 100
total_required = 10000

papers = []
seen_titles = set()   # to remove duplicates

for query in queries:
    print(f"\nCollecting papers for query: {query}")
    offset = 0

    while len(papers) < total_required:

        params = {
            "query": query,
            "limit": limit_per_request,
            "offset": offset,
            "fields": "title,abstract,year"
        }

        response = requests.get(API_URL, headers=headers, params=params)

        if response.status_code == 429:
            print("Rate limit reached... waiting 10 seconds")
            time.sleep(10)
            continue

        if response.status_code != 200:
            print("Request failed:", response.status_code)
            break

        data = response.json()

        if "data" not in data or len(data["data"]) == 0:
            print("No more data available for this query.")
            break

        for paper in data["data"]:
            title = paper.get("title", "")
            abstract = paper.get("abstract", "")
            year = paper.get("year", "")

            # Remove duplicates by title
            if title not in seen_titles and abstract:
                papers.append({
                    "title": title,
                    "abstract": abstract,
                    "year": year
                })
                seen_titles.add(title)

            if len(papers) >= total_required:
                break

        offset += limit_per_request
        print("Collected so far:", len(papers))

        time.sleep(2)

    if len(papers) >= total_required:
        break


# Create dataframe
df = pd.DataFrame(papers)

# Final safety cleaning
df = df[df["abstract"] != ""]

# Save CSV
df.to_csv("raw_papers.csv", index=False)

print("✅ Data collection complete")
print("Rows collected:", len(df))

df.head()


Collected so far: 63
Collected so far: 140
Rate limit reached... waiting 10 seconds
Rate limit reached... waiting 10 seconds
Rate limit reached... waiting 10 seconds
Rate limit reached... waiting 10 seconds
Collected so far: 212
Rate limit reached... waiting 10 seconds
Collected so far: 240
Rate limit reached... waiting 10 seconds
Collected so far: 270
Rate limit reached... waiting 10 seconds
Rate limit reached... waiting 10 seconds
Rate limit reached... waiting 10 seconds
Collected so far: 321
Rate limit reached... waiting 10 seconds
Rate limit reached... waiting 10 seconds
Collected so far: 339
Collected so far: 345
Rate limit reached... waiting 10 seconds
Rate limit reached... waiting 10 seconds
Rate limit reached... waiting 10 seconds
Rate limit reached... waiting 10 seconds
Rate limit reached... waiting 10 seconds
Collected so far: 385
Rate limit reached... waiting 10 seconds
Collected so far: 413
Rate limit reached... waiting 10 seconds
Rate limit reached... waiting 10 seconds
R

,title,abstract,year
0,"Machine Learning: Algorithms, Real-World Appli...",In the current age of the Fourth Industrial Re...,2021
1,Fashion-MNIST: a Novel Image Dataset for Bench...,"We present Fashion-MNIST, a new dataset compri...",2017
2,A Survey on Bias and Fairness in Machine Learning,With the widespread use of artificial intellig...,2019
3,Towards A Rigorous Science of Interpretable Ma...,"As machine learning systems become ubiquitous,...",2017
4,TensorFlow: A system for large-scale machine l...,TensorFlow is a machine learning system that o...,2016


# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

In [2]:

# QUESTION 2
# Install library
!pip install nltk

import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download required datasets
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# ------------------------------
# Load dataset from Question 1
# ------------------------------
df = pd.read_csv("raw_papers.csv")

# ✅ Remove rows where abstract is missing
df = df.dropna(subset=["abstract"])

# ------------------------------
# Initialize NLP tools
# ------------------------------
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# ------------------------------
# Cleaning Function
# ------------------------------
def clean_text(text):

    text = str(text)

    # (4) Lowercase all text
    text = text.lower()

    # (1) Remove noise (special characters & punctuation)
    text = re.sub(r'[^a-z\s]', ' ', text)

    # (2) Remove numbers
    text = re.sub(r'\d+', '', text)

    # Tokenization
    words = text.split()

    # (3) Remove stopwords
    words = [w for w in words if w not in stop_words]

    # (5) Stemming
    words = [stemmer.stem(w) for w in words]

    # (6) Lemmatization
    words = [lemmatizer.lemmatize(w) for w in words]

    return " ".join(words)

# ------------------------------
# Apply Cleaning
# ------------------------------
df["clean_text"] = df["abstract"].apply(clean_text)

# ------------------------------
# Save Cleaned Dataset
# ------------------------------
df.to_csv("cleaned_papers.csv", index=False)

print("✅ Text cleaning completed successfully!")
print("Rows after cleaning:", len(df))

# ------------------------------
# Show Output
# ------------------------------
df[["abstract", "clean_text"]].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


✅ Text cleaning completed successfully!
Rows after cleaning: 1033


,abstract,clean_text
0,In the current age of the Fourth Industrial Re...,current age fourth industri revolut ir industr...
1,"We present Fashion-MNIST, a new dataset compri...",present fashion mnist new dataset compris x gr...
2,With the widespread use of artificial intellig...,widespread use artifici intellig ai system app...
3,"As machine learning systems become ubiquitous,...",machin learn system becom ubiquit surg interes...
4,TensorFlow is a machine learning system that o...,tensorflow machin learn system oper larg scale...


# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [3]:
# Fix spaCy installation completely
!pip install -U spacy==3.7.4

# Install model directly (bypass broken downloader)
!pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 41.2 MB/s eta 0:00:00


In [4]:
import spacy
nlp = spacy.load("en_core_web_sm")
print("✅ spaCy model loaded successfully")

✅ spaCy model loaded successfully


In [5]:
import os
os.listdir()

['.config', 'raw_papers.csv', 'cleaned_papers.csv', 'sample_data']

In [6]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [7]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [8]:
# ==========================================
# SYNTAX & STRUCTURE ANALYSIS
# ==========================================

# Install required libraries
!pip install -q spacy nltk
!pip install -q https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl

import pandas as pd
import spacy
import nltk
from nltk import word_tokenize, pos_tag, RegexpParser
from collections import Counter

# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')

# Load dataset
df = pd.read_csv("cleaned_papers.csv")
texts = df["clean_text"].dropna().astype(str).tolist()

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# =====================================================
# (1) POS TAGGING
# =====================================================

noun = verb = adj = adv = 0

for text in texts:
    tokens = word_tokenize(text)
    tags = pos_tag(tokens)

    for word, tag in tags:
        if tag.startswith("NN"):
            noun += 1
        elif tag.startswith("VB"):
            verb += 1
        elif tag.startswith("JJ"):
            adj += 1
        elif tag.startswith("RB"):
            adv += 1

print("\n========== POS TAG COUNTS ==========")
print("Total Nouns:", noun)
print("Total Verbs:", verb)
print("Total Adjectives:", adj)
print("Total Adverbs:", adv)


# =====================================================
# (2) CONSTITUENCY PARSING
# =====================================================

print("\n========== CONSTITUENCY PARSING ==========")

for text in texts:
    tokens = word_tokenize(text)
    tags = pos_tag(tokens)

    grammar = """
    NP: {<DT>?<JJ>*<NN.*>}
    VP: {<VB.*><NP|PP>*}
    """

    parser = RegexpParser(grammar)
    tree = parser.parse(tags)

    print("\nSentence:")
    print(text)
    print("\nConstituency Tree:")
    print(tree)

# =====================================================
# DEPENDENCY PARSING
# =====================================================

print("\n========== DEPENDENCY PARSING ==========")

for text in texts:
    doc = nlp(text)
    print("\nSentence:")
    print(text)

    for token in doc:
        print(token.text, "->", token.dep_, "->", token.head.text)


# Example explanation (one sentence)
example_sentence = texts[0]
print("\n========== EXPLANATION EXAMPLE ==========")
print("Example Sentence:", example_sentence)

print("""
Constituency Parsing:
It groups words into hierarchical phrase structures such as
noun phrases (NP) and verb phrases (VP).
It shows how words combine to form larger grammatical units.

Dependency Parsing:
It shows grammatical relationships between words.
Each word depends on a head word.
For example, a subject depends on a verb.
""")


# =====================================================
# (3) NAMED ENTITY RECOGNITION
# =====================================================

entity_counts = Counter()

for text in texts:
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG", "GPE", "DATE", "PRODUCT"]:
            entity_counts[ent.label_] += 1

print("\n========== NAMED ENTITY COUNTS ==========")
for entity, count in entity_counts.items():
    print(entity, ":", count)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


Streaming output truncated to the last 5000 lines.
articl -> npadvmod -> extract
report -> dobj -> extract
random -> amod -> control
control -> compound -> trial
trial -> compound -> methodsexact
rct -> compound -> methodsexact
methodsexact -> nsubj -> consist
consist -> conj -> extract
two -> nummod -> inform
part -> compound -> inform
inform -> dobj -> consist
extract -> xcomp -> consist
ie -> nmod -> fragment
engin -> compound -> search
search -> compound -> articl
articl -> compound -> fragment
text -> compound -> fragment
fragment -> dobj -> extract
best -> amod -> interfac
describ -> compound -> browser
trial -> compound -> characterist
characterist -> compound -> browser
web -> compound -> browser
browser -> compound -> base
base -> compound -> user
user -> compound -> interfac
interfac -> nsubj -> allow
allow -> ROOT -> allow
human -> amod -> review
review -> nsubj -> assess
assess -> compound -> modifi
modifi -> nsubj -> suggest
suggest -> ccomp -> allow
select -> acomp -> sug

# **Following Questions must answer using AI assitance**

#Question 4 (20 points).

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


Github MarketPlace page:
https://github.com/marketplace?type=actions

In [9]:
# ==============================================
# GITHUB ACTIONS DATA COLLECTION (REST API)
# + DATA QUALITY + PREPROCESSING
# ==============================================

!pip install -q requests pandas nltk

import requests
import pandas as pd
import time
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# ==============================================
# PART 1 — DATA COLLECTION (Pagination handled)
# ==============================================

data = []
target_count = 1000
page = 1

while len(data) < target_count:

    print(f"Collecting page {page}...")

    url = f"https://api.github.com/search/repositories?q=topic:github-action&per_page=100&page={page}"
    response = requests.get(url)

    if response.status_code != 200:
        print("Request failed. Status:", response.status_code)
        break

    items = response.json().get("items", [])

    if not items:
        print("No more results.")
        break

    for repo in items:
        data.append({
            "Product Name": repo.get("name"),
            "Description": repo.get("description"),
            "URL": repo.get("html_url"),
            "Page Number": page
        })

    page += 1
    time.sleep(2)  # respectful delay

df = pd.DataFrame(data).head(1000)

df.to_csv("github_marketplace_raw.csv", index=False)
print("✅ Data collected:", len(df))

# PART 2 — DATA QUALITY
print("Initial shape:", df.shape)

df.drop_duplicates(inplace=True)
df.dropna(subset=["Product Name", "Description", "URL"], inplace=True)

print("After data quality cleaning:", df.shape)


# PREPROCESSING


stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)         # remove html
    text = re.sub(r'[^a-z\s]', '', text)     # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return " ".join(tokens)

df["Cleaned Description"] = df["Description"].apply(preprocess)

df.to_csv("github_marketplace_cleaned.csv", index=False)

print("✅ Preprocessing completed.")
df.head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


✅ Data collected: 1000
Initial shape: (1000, 4)
After data quality cleaning: (981, 4)
✅ Preprocessing completed.


,Product Name,Description,URL,Page Number,Cleaned Description
0,metrics,📊 An infographics generator with 30+ plugins a...,https://github.com/lowlighter/metrics,1,infographics generator plugins option display ...
1,zotero-arxiv-daily,Recommend new arxiv papers of your interest da...,https://github.com/TideDra/zotero-arxiv-daily,1,recommend new arxiv paper interest daily accor...
2,github-pages-deploy-action,🚀 Automatically deploy your project to GitHub ...,https://github.com/JamesIves/github-pages-depl...,1,automatically deploy project github page using...
3,action-tmate,Debug your GitHub Actions via SSH by using tma...,https://github.com/mxschmitt/action-tmate,1,debug github action via ssh using tmate get ac...
4,github-profile-summary-cards,A tool to generate your GitHub summary card fo...,https://github.com/vn7n24fzkq/github-profile-s...,1,tool generate github summary card profile readme


In [13]:
print("\n========== DATA QUALITY REPORT ==========\n")

print("Initial dataset shape:", df.shape)

# 1️⃣ Missing Values
print("\nMissing values per column:")
print(df.isnull().sum())

# 2️⃣ Remove duplicates
duplicate_count = df.duplicated().sum()
print("\nDuplicate rows found:", duplicate_count)
df.drop_duplicates(inplace=True)

# 3️⃣ Remove rows with missing critical fields
df.dropna(subset=["Tweet ID", "Username", "Text"], inplace=True)

# 4️⃣ Remove empty text
df = df[df["Text"].str.strip() != ""]

# 5️⃣ Remove very short tweets (quality filter)
df = df[df["Text"].str.len() > 10]

print("\nFinal dataset shape after cleaning:", df.shape)

print("\n========== DATA QUALITY COMPLETE ==========\n")


========== DATA QUALITY REPORT ==========

Initial dataset shape: (5, 4)

Missing values per column:
Tweet ID        0
Username        0
Text            0
Cleaned Text    0
dtype: int64

Duplicate rows found: 0

Final dataset shape after cleaning: (5, 4)

========== DATA QUALITY COMPLETE ==========



#Question 5 (20 points)

PART 1:
Web Scrape  tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.)
The extracted data includes the tweet ID, username, and text.

Part 2:
Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.


**Note**

1.   Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2.   Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.


In [10]:
import pandas as pd

# Simulated API-like tweet data (ML/AI related)
tweets = [
    {"Tweet ID": 1, "Username": "ai_researcher", "Text": "Machine learning is transforming healthcare using deep learning models."},
    {"Tweet ID": 2, "Username": "tech_guru", "Text": "Artificial intelligence and neural networks are advancing rapidly."},
    {"Tweet ID": 3, "Username": "ml_engineer", "Text": "AI applications in natural language processing are impressive."},
    {"Tweet ID": 4, "Username": "data_scientist", "Text": "Deep learning improves image recognition accuracy significantly."},
    {"Tweet ID": 5, "Username": "ai_news", "Text": "Machine learning algorithms are used in self-driving cars."},
]

df = pd.DataFrame(tweets)

print("Tweets Collected:", len(df))
df

Tweets Collected: 5


,Tweet ID,Username,Text
0,1,ai_researcher,Machine learning is transforming healthcare us...
1,2,tech_guru,Artificial intelligence and neural networks ar...
2,3,ml_engineer,AI applications in natural language processing...
3,4,data_scientist,Deep learning improves image recognition accur...
4,5,ai_news,Machine learning algorithms are used in self-d...


In [11]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]

    return " ".join(tokens)

df["Cleaned Text"] = df["Text"].apply(clean_text)

print("Initial Shape:", df.shape)

df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

print("After Cleaning Shape:", df.shape)

df.to_csv("cleaned_tweets.csv", index=False)

print("File saved as cleaned_tweets.csv")

from google.colab import files
files.download("cleaned_tweets.csv")

Initial Shape: (5, 4)
After Cleaning Shape: (5, 4)
File saved as cleaned_tweets.csv


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
# CLEANING raw_papers.csv AND SAVING cleaned_papers.csv

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Load raw data
df = pd.read_csv("raw_papers.csv")

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()                         # Lowercase
    text = re.sub(r"[^a-z\s]", "", text)             # Remove special characters & numbers
    text = re.sub(r"\s+", " ", text).strip()         # Remove extra spaces

    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]  # Remove stopwords

    stemmed = [stemmer.stem(w) for w in tokens]           # Stemming
    lemmatized = [lemmatizer.lemmatize(w) for w in stemmed]  # Lemmatization

    return " ".join(lemmatized)

# Apply cleaning to abstract column
df["cleaned_abstract"] = df["abstract"].apply(clean_text)

# Data Quality Checks
df.drop_duplicates(inplace=True)
df.dropna(subset=["cleaned_abstract"], inplace=True)

# Save cleaned CSV
df.to_csv("cleaned_papers.csv", index=False)

print("✅ Cleaned file saved as cleaned_papers.csv")
print("Rows after cleaning:", len(df))

df.head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


✅ Cleaned file saved as cleaned_papers.csv
Rows after cleaning: 1033


,title,abstract,year,cleaned_abstract
0,"Machine Learning: Algorithms, Real-World Appli...",In the current age of the Fourth Industrial Re...,2021,current age fourth industri revolut ir industr...
1,Fashion-MNIST: a Novel Image Dataset for Bench...,"We present Fashion-MNIST, a new dataset compri...",2017,present fashionmnist new dataset compris x gra...
2,A Survey on Bias and Fairness in Machine Learning,With the widespread use of artificial intellig...,2019,widespread use artifici intellig ai system app...
3,Towards A Rigorous Science of Interpretable Ma...,"As machine learning systems become ubiquitous,...",2017,machin learn system becom ubiquit surg interes...
4,TensorFlow: A system for large-scale machine l...,TensorFlow is a machine learning system that o...,2016,tensorflow machin learn system oper larg scale...


In [15]:
from google.colab import files
files.download("cleaned_papers.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

The hardest part was the API authentication and access. These errors like unauthorized responses and rate limits had to be problem-solved, read the documentation, and reformatted the implementation. The detail of the understanding also required to comprehend how APIs make their answers in the clothe of JSON and how to extract the suitable fields like ID, username and text in specific.

The other issue was to ensure that the right data cleaning was obtained. There was the need to remove the special characters and URLs and keep meaningful information. Such screening of data quality like the removal of duplicates and the establishment of missing values highlighted the necessity of having clean and homogeneous sets of data to work with.

Another aspect I particularly liked was the use of text preprocessing such as tokenization and normalization. The possibility to work with raw text and transform it into formatted and clean data that could be analyzed were also rewarding and confirmed theoretical information obtained at the classroom.

Overall the given time was decent, however, there were also some negative surprises, the presence of which was associated with API restrictions that resulted in the further extension of work. However, the assignment has assisted me in developing my expertise of data collection and preprocessing procedures.